**作用：**对历史消息列表进行 `摘要&总结` ，达到 `压缩上下文` 的效果。

**原理：**在 `达到触发条件` 时，调用大模型对历史消息进行摘要， `将摘要的结果作为HumanMessage` ，放到消息列表最开始的位置。

初始化模型

In [6]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
# 优先加载大模型
load_dotenv(override=True)

#设置最大的输入token
custom_profile = {
    "max_input_tokens": 128_000,
}

#初始化大模型
model_openai = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    profile=custom_profile,
)

测试trigger、keep参数

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思")
]

agent = create_agent(
    model="deepseek-flash",
    middleware=[
        SummarizationMiddleware(
            model=model_openai,
            trigger=[
                ("tokens", 100),
                ("messages", 6),
                ("fraction", 0.001)
            ],
            keep=("messages", 2)
        )
    ]
)

response = agent.invoke({
    "messages": messages,
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT
Initial greeting and conversational setup. The user's immediate goal is to introduce themselves and establish a working rapport with the AI. No specific task, project, or problem-solving objective has been assigned yet.

## SUMMARY
- **User Identity:** 老王 (Lao Wang)
- **AI Identity/Persona:** 小王 (Xiao Wang), consistent with the system prompt defining the AI as a friendly assistant.
- **Interaction Status:** Polite mutual introductions completed successfully. No actionable requests, technical discussions, decisions, or rejected options exist. The session is currently at a pre-task baseline awaiting user direction.

## ARTIFACTS
None.

## NEXT STEPS
Standby for 老王's first explicit request or task specification. Upon receipt, initiate analysis, planning, or execution relevant to their stated objective.
================================== Ai Mes

测试summary_prompt参数

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.messages import SystemMessage, HumanMessage, AIMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
# 优先加载大模型
load_dotenv(override=True)

#设置最大的输入token
custom_profile = {
    "max_input_tokens": 1_000_000,
}

# 初始化用于摘要的模型 (建议使用较小/便宜的模型)
summarizer_model = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    profile=custom_profile,
)

# 主模型
main_model = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

messages = [
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思，你是谁？")
]

agent = create_agent(
    model=main_model,
    middleware=[
        SummarizationMiddleware(
            model=summarizer_model,
            trigger=[
                ("tokens", 100),
                ("messages", 6),
                ("fraction", 0.001)
            ],
            keep=("messages", 2),
            summary_prompt="对历史消息摘要，消息列表如下\n{messages}"
        )
    ]
)

response = agent.invoke({
    "messages": messages,
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

以下是该段历史消息的结构化摘要，便于上下文记忆或后续调用：

**📝 对话摘要**  
当前处于**初始寒暄阶段**。用户自称为“老王”，AI以“小王”身份回应，双方已完成礼貌问候。结合系统设定，已明确“友好型AI助手”的角色定位，交互氛围轻松融洽。后续对话可直接沿用该称呼与语气继续。

**🔑 关键要素提取**
- **角色映射**：用户 → `老王`｜AI → `小王`（友好助手）
- **交互状态**：已完成首次自我介绍与礼仪性寒暄
- **语气基调**：礼貌、轻松、日常对话
- **上下文建议**：后续回复可自然延续“小王”称呼，无需重复背景说明

如需将此摘要用于 Prompt 注入或记忆管理，可直接复制上方加粗部分使用。
================================== Ai Message ==================================

你高兴得太早了
================================ Human Message =================================

呵呵，你什么意思，你是谁？
================================== Ai Message ==================================

哈哈，老王别多想～我就是咱们之前对好的“小王”，你的AI小助手。刚才那段摘要纯粹是为了让之后的聊天更顺滑、不用每次都重新认人，真没别的意思哈。  
有啥问题、想聊的、或者需要搭把手的，随时吱声，我在这儿稳稳的。今天老王准备忙点啥？
